#### Chatbot Evaluation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [64]:
from langsmith import Client

client = Client()

dataset_name = "Electric Vehicle Evaluation" 
dataset = client.create_dataset(dataset_name)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is regenerative braking?"},
            "outputs": {"answer": "A mechanism that recovers kinetic energy during braking and stores it as electricity in the battery."},
        },
        {
            "inputs": {"question": "What are the main components of an EV powertrain?"},
            "outputs": {"answer": "The battery pack, electric motor, and the onboard charger/inverter."},
        },
        {
            "inputs": {"question": "What does 'Range Anxiety' refer to?"},
            "outputs": {"answer": "The fear that a vehicle has insufficient energy storage to cover the distance needed to reach its destination."},
        }
    ]
)

{'example_ids': ['7f5d933d-4d48-405f-b6ea-72334a85aab5',
  '24f73d88-647f-4334-951b-709fb684dcbc',
  '41fdb470-c4d8-4bbb-a8d7-a9a0a9813077'],
 'count': 3,
 'as_of': '2026-03-04T12:02:59.705819661Z'}

#### Define Metrics (LLM As A Judge)


In [ ]:
import openai
from langsmith import wrappers
 
openai_client=wrappers.wrap_openai(openai.OpenAI())

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
      response=openai_client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0,
            messages=[
                  {"role":"system","content":eval_instructions},
                  {"role":"user","content":user_content}
            ]
      ).choices[0].message.content

      return response == "CORRECT"

In [66]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

#### Run Evaluations

In [67]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "gpt-3.5-turbo", instructions: str = default_instructions) -> str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [68]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [69]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-3.5-turbo-chatbot"
)

View the evaluation results for experiment: 'openai-3.5-turbo-chatbot-10113e4a' at:
https://smith.langchain.com/o/ed875a17-f1db-4f8d-8ba4-8eed0a9c26b2/datasets/58e88bea-781f-4225-a9ab-2c6bc7a4a86f/compare?selectedSessions=8483f334-784b-47e6-93c5-1091b72e9239




3it [00:06,  2.31s/it]


In [70]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="gpt-3.5-turbo")}

In [71]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="openai-3.5-turbo-chatbot"
)

View the evaluation results for experiment: 'openai-3.5-turbo-chatbot-f1c435ab' at:
https://smith.langchain.com/o/ed875a17-f1db-4f8d-8ba4-8eed0a9c26b2/datasets/58e88bea-781f-4225-a9ab-2c6bc7a4a86f/compare?selectedSessions=af1e7e2d-dfae-44ed-b0c3-904df2ece708




3it [00:05,  1.90s/it]
